# Predicting Boston Housing Prices: A Regression Model Comparison

**Author:** Thomas Potts  
**Tools:** Python, pandas, scikit-learn, matplotlib, NumPy

## Overview

This project explores multiple regression approaches to predict median home values in the Boston housing dataset (506 observations, 13 features). The analysis has three parts:

1. **Model Comparison** — Evaluate KNN, Decision Tree, and Random Forest regressors using GridSearchCV with 5-fold cross-validation to identify the best-performing model.
2. **Linear Regression from Scratch** — Implement gradient descent-based linear regression and benchmark it against scikit-learn's implementation.
3. **KNN Regression from Scratch** — Implement a KNN regressor from scratch and verify it produces identical results to scikit-learn.

Models are evaluated on MSE, MAE, and MAPE.

## Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from time import time
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression as sk_LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler

## Data Loading & Exploration

The dataset contains 506 observations across 14 variables. The target variable is **MEDV** (median value of owner-occupied homes in $1,000s). Features include crime rate, land zoning, proximity to employment centers, tax rates, and other neighborhood characteristics.

In [ ]:
housing = pd.read_csv('boston_housing_missing_values.csv')
print(f"Shape: {housing.shape}")
housing.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1.0,296.0,15.3,NaN,4.98,24.0
1,0.02731,0.0,7.07,0.0,0.469,NaN,78.9,4.9671,2.0,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2.0,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3.0,222.0,18.7,394.63,2.94,33.4
4,NaN,0.0,2.18,0.0,0.458,7.147,NaN,6.0622,3.0,222.0,18.7,396.90,5.33,36.2


### Feature Descriptions

| Feature | Description |
|---------|-------------|
| CRIM | Per capita crime rate by town |
| ZN | Proportion of residential land zoned for lots over 25,000 sq.ft. |
| INDUS | Proportion of non-retail business acres per town |
| CHAS | Charles River dummy variable (1 if tract bounds river; 0 otherwise) |
| NOX | Nitric oxides concentration (parts per 10 million) |
| RM | Average number of rooms per dwelling |
| AGE | Proportion of owner-occupied units built prior to 1940 |
| DIS | Weighted distances to five Boston employment centres |
| RAD | Index of accessibility to radial highways |
| TAX | Full-value property-tax rate per $10,000 |
| PTRATIO | Pupil-teacher ratio by town |
| B | 1000(Bk - 0.63)² where Bk is the proportion of Black residents by town |
| LSTAT | Percentage lower status of the population |
| MEDV | Median value of owner-occupied homes in $1,000s (target) |

## Data Cleaning: Handling Missing Values

The raw dataset contains missing values. I use mean imputation via scikit-learn's `SimpleImputer` to fill gaps — a straightforward approach appropriate for this dataset size.

In [ ]:
print(f"Total missing values before imputation: {housing.isnull().sum().sum()}")

imputer = SimpleImputer(strategy='mean')
housing_imputed = imputer.fit_transform(housing)
housing = pd.DataFrame(housing_imputed, columns=housing.columns)

print(f"Total missing values after imputation: {housing.isnull().sum().sum()}")

## Train/Test Split

Split 80/20 with a fixed random state for reproducibility. Features (X) are the first 13 columns; target (Y) is MEDV.

In [ ]:
X = housing.iloc[:, 0:13].values
Y = housing.iloc[:, 13].values

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)

print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test set:     {X_test.shape[0]} samples")

---
## Part 1: Model Comparison with GridSearchCV

I compare three regression models using 5-fold cross-validation with negative MSE as the scoring metric. GridSearchCV automates hyperparameter tuning for each model.

### KNN Regressor

Tuning `n_neighbors` across a range of values. Feature scaling with MinMaxScaler is applied since KNN is distance-based.

In [ ]:
n_list = np.arange(1, 100, 10)
param_grid = {'n_neighbors': n_list}

gs_KNN = GridSearchCV(
    estimator=KNeighborsRegressor(),
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=5
)

start_time = time()
gs_KNN.fit(X_train, Y_train)
end_time = time()
KNN_time = end_time - start_time

print(f"Best n_neighbors: {gs_KNN.best_params_['n_neighbors']}")
print(f"GridSearchCV time: {KNN_time:.3f}s")

In [ ]:
# Apply MinMaxScaler for KNN (distance-based model benefits from scaling)
scaler = MinMaxScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

KNN_model_best = gs_KNN.best_estimator_
KNN_model_best.fit(X_train_s, Y_train)
Y_test_pred = KNN_model_best.predict(X_test_s)

KNN_MSE = np.mean((Y_test - Y_test_pred)**2)
KNN_MAE = np.mean(np.abs(Y_test - Y_test_pred))
KNN_MAPE = np.mean(np.abs((Y_test - Y_test_pred) / Y_test))

print(f"KNN — MSE: {KNN_MSE:.2f}, MAE: {KNN_MAE:.2f}, MAPE: {KNN_MAPE:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(Y_test, Y_test, '-r', label='Perfect prediction')
ax.plot(Y_test, Y_test_pred, '.b', alpha=0.7, label='Predicted')
ax.set_xlabel('Actual MEDV ($1,000s)')
ax.set_ylabel('Predicted MEDV ($1,000s)')
ax.set_title('KNN Regressor: Predicted vs. Actual')
ax.legend()
plt.tight_layout()
plt.show()

### Decision Tree Regressor

Tuning `max_depth` to control tree complexity and prevent overfitting.

In [ ]:
max_depth_list = np.arange(1, 100, 5)
param_grid = {'max_depth': max_depth_list, 'random_state': [0]}

gs_DT = GridSearchCV(
    estimator=DecisionTreeRegressor(),
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=5
)

start_time = time()
gs_DT.fit(X_train, Y_train)
end_time = time()
DT_time = end_time - start_time

print(f"Best max_depth: {gs_DT.best_params_['max_depth']}")
print(f"GridSearchCV time: {DT_time:.3f}s")

In [ ]:
DT_model_best = gs_DT.best_estimator_
DT_model_best.fit(X_train, Y_train)
Y_test_pred = DT_model_best.predict(X_test)

DT_MSE = np.mean((Y_test - Y_test_pred)**2)
DT_MAE = np.mean(np.abs(Y_test - Y_test_pred))
DT_MAPE = np.mean(np.abs((Y_test - Y_test_pred) / Y_test))

print(f"Decision Tree — MSE: {DT_MSE:.2f}, MAE: {DT_MAE:.2f}, MAPE: {DT_MAPE:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(Y_test, Y_test, '-r', label='Perfect prediction')
ax.plot(Y_test, Y_test_pred, '.b', alpha=0.7, label='Predicted')
ax.set_xlabel('Actual MEDV ($1,000s)')
ax.set_ylabel('Predicted MEDV ($1,000s)')
ax.set_title('Decision Tree: Predicted vs. Actual')
ax.legend()
plt.tight_layout()
plt.show()

### Random Forest Regressor

Tuning `max_depth` with an ensemble of 10 trees. Random Forest reduces the variance problem of a single decision tree by averaging predictions across multiple trees.

In [ ]:
param_grid = {
    'max_depth': np.arange(1, 100, 5),
    'n_estimators': [10],
    'random_state': [0]
}

gs_RF = GridSearchCV(
    estimator=RandomForestRegressor(),
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=5
)

start_time = time()
gs_RF.fit(X_train, Y_train)
end_time = time()
RF_time = end_time - start_time

print(f"Best max_depth: {gs_RF.best_params_['max_depth']}")
print(f"GridSearchCV time: {RF_time:.3f}s")

In [ ]:
RF_model_best = gs_RF.best_estimator_
RF_model_best.fit(X_train, Y_train)
Y_test_pred = RF_model_best.predict(X_test)

RF_MSE = np.mean((Y_test - Y_test_pred)**2)
RF_MAE = np.mean(np.abs(Y_test - Y_test_pred))
RF_MAPE = np.mean(np.abs((Y_test - Y_test_pred) / Y_test))

print(f"Random Forest — MSE: {RF_MSE:.2f}, MAE: {RF_MAE:.2f}, MAPE: {RF_MAPE:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(Y_test, Y_test, '-r', label='Perfect prediction')
ax.plot(Y_test, Y_test_pred, '.b', alpha=0.7, label='Predicted')
ax.set_xlabel('Actual MEDV ($1,000s)')
ax.set_ylabel('Predicted MEDV ($1,000s)')
ax.set_title('Random Forest: Predicted vs. Actual')
ax.legend()
plt.tight_layout()
plt.show()

### Model Comparison Summary

Comparing all three models across error metrics and computation time.

In [ ]:
results = pd.DataFrame({
    'Model': ['KNN', 'Decision Tree', 'Random Forest'],
    'GridSearchCV Time (s)': [KNN_time, DT_time, RF_time],
    'MSE': [KNN_MSE, DT_MSE, RF_MSE],
    'MAE': [KNN_MAE, DT_MAE, RF_MAE],
    'MAPE': [KNN_MAPE, DT_MAPE, RF_MAPE]
})
results.style.format({
    'GridSearchCV Time (s)': '{:.3f}',
    'MSE': '{:.2f}',
    'MAE': '{:.2f}',
    'MAPE': '{:.3f}'
})

**Key Takeaway:** Random Forest achieved the lowest MAE and MAPE, making it the best performer on this dataset. KNN was competitive after feature scaling. Decision Tree had the highest error, likely due to overfitting despite depth tuning.

---
## Part 2: Linear Regression — Implemented from Scratch

To deepen my understanding of how linear regression works under the hood, I implemented gradient descent optimization from scratch. The model learns weights **w** and bias **b** to minimize mean squared error:

$$f(x) = \sum_{n=0}^{M-1} w[n] \cdot x[n] + b$$

Gradients are computed analytically and applied over 1,000,000 iterations with a learning rate of 1e-6.

In [ ]:
class MyLinearRegression:
    def __init__(self, learning_rate, max_iter):
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.w = None
        self.b = None

    def fit(self, X, Y):
        N, M = X.shape
        self.w = np.random.randn(M)
        self.b = np.random.randn()
        for i in range(self.max_iter):
            Y_pred = X @ self.w + self.b
            dL_dw = -2 * (X * (Y - Y_pred)[:, None]).mean(axis=0)
            dL_db = -2 * (Y - Y_pred).mean()
            self.w -= self.learning_rate * dL_dw
            self.b -= self.learning_rate * dL_db

    def predict(self, X):
        return X @ self.w + self.b

In [ ]:
t0 = time()
my_model = MyLinearRegression(learning_rate=1e-6, max_iter=1000000)
my_model.fit(X_train, Y_train)
t1 = time()
print(f"Training time: {t1-t0:.2f}s")

### Evaluating the Custom Implementation

Comparing my gradient descent model against scikit-learn's closed-form Linear Regression on the same test set.

In [ ]:
# Scikit-learn baseline
sk_model = sk_LinearRegression()
sk_model.fit(X_train, Y_train)
sk_pred = sk_model.predict(X_test)
sk_MSE = np.mean((Y_test - sk_pred)**2)
sk_MAE = np.mean(np.abs(Y_test - sk_pred))
sk_MAPE = np.mean(np.abs(Y_test - sk_pred) / Y_test)

# Custom implementation
my_pred = my_model.predict(X_test)
my_MSE = np.mean((Y_test - my_pred)**2)
my_MAE = np.mean(np.abs(Y_test - my_pred))
my_MAPE = np.mean(np.abs(Y_test - my_pred) / Y_test)

comparison = pd.DataFrame({
    'Model': ['sklearn LinearRegression', 'MyLinearRegression (gradient descent)'],
    'MSE': [sk_MSE, my_MSE],
    'MAE': [sk_MAE, my_MAE],
    'MAPE': [sk_MAPE, my_MAPE]
})
comparison

In [ ]:
# Measure prediction divergence between the two implementations
divergence = np.mean(np.abs(my_pred - sk_pred) / np.abs(sk_pred.mean()))
print(f"Mean relative divergence from sklearn: {divergence:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(Y_test, Y_test, '-r', label='Perfect prediction')
axes[0].plot(Y_test, sk_pred, '.b', alpha=0.7, label='Predicted')
axes[0].set_xlabel('Actual MEDV ($1,000s)')
axes[0].set_ylabel('Predicted MEDV ($1,000s)')
axes[0].set_title('sklearn LinearRegression')
axes[0].legend()

axes[1].plot(Y_test, Y_test, '-r', label='Perfect prediction')
axes[1].plot(Y_test, my_pred, '.b', alpha=0.7, label='Predicted')
axes[1].set_xlabel('Actual MEDV ($1,000s)')
axes[1].set_ylabel('Predicted MEDV ($1,000s)')
axes[1].set_title('MyLinearRegression (Gradient Descent)')
axes[1].legend()

plt.tight_layout()
plt.show()

The custom gradient descent implementation converges close to sklearn's closed-form solution, with a relative divergence under 10%. The remaining gap is expected — gradient descent is an iterative approximation, while sklearn uses the normal equation for an exact solution.

---
## Part 3: KNN Regression — Implemented from Scratch

I also implemented KNN regression from scratch to understand the algorithm at a fundamental level. The predict method computes Euclidean distances from each test point to all training points, identifies the K nearest neighbors, and returns the mean of their target values.

In [ ]:
class MyKNNRegressor:
    def __init__(self, n_neighbors):
        self.n_neighbors = n_neighbors
        self.X_train = None
        self.Y_train = None

    def fit(self, X, Y):
        self.X_train = X
        self.Y_train = Y

    def predict(self, X):
        preds = []
        for x in X:
            dists = np.linalg.norm(self.X_train - x, axis=1)
            idx = np.argsort(dists)[:self.n_neighbors]
            preds.append(self.Y_train[idx].mean())
        return np.array(preds)

In [ ]:
t0 = time()
my_knn = MyKNNRegressor(n_neighbors=11)
my_knn.fit(X_train, Y_train)
t1 = time()
print(f"Fit time: {t1-t0:.6f}s (lazy learner — just stores data)")

### Verifying Correctness Against scikit-learn

In [ ]:
# sklearn baseline (no scaling, to match custom implementation)
sk_knn = KNeighborsRegressor(n_neighbors=11)
sk_knn.fit(X_train, Y_train)
sk_knn_pred = sk_knn.predict(X_test)

# Custom implementation
my_knn_pred = my_knn.predict(X_test)

my_knn_MSE = np.mean((Y_test - my_knn_pred)**2)
my_knn_MAE = np.mean(np.abs(Y_test - my_knn_pred))
my_knn_MAPE = np.mean(np.abs(Y_test - my_knn_pred) / Y_test)

max_diff = np.max(np.abs(my_knn_pred - sk_knn_pred))
print(f"Custom KNN — MSE: {my_knn_MSE:.2f}, MAE: {my_knn_MAE:.2f}, MAPE: {my_knn_MAPE:.3f}")
print(f"Max absolute difference from sklearn: {max_diff}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(Y_test, Y_test, '-r', label='Perfect prediction')
axes[0].plot(Y_test, sk_knn_pred, '.b', alpha=0.7, label='Predicted')
axes[0].set_xlabel('Actual MEDV ($1,000s)')
axes[0].set_ylabel('Predicted MEDV ($1,000s)')
axes[0].set_title('sklearn KNeighborsRegressor')
axes[0].legend()

axes[1].plot(Y_test, Y_test, '-r', label='Perfect prediction')
axes[1].plot(Y_test, my_knn_pred, '.b', alpha=0.7, label='Predicted')
axes[1].set_xlabel('Actual MEDV ($1,000s)')
axes[1].set_ylabel('Predicted MEDV ($1,000s)')
axes[1].set_title('MyKNNRegressor')
axes[1].legend()

plt.tight_layout()
plt.show()

The custom KNN implementation produces **identical predictions** to scikit-learn (max difference = 0.0), confirming correctness.

---
## Conclusion

| Approach | Best MSE | Notes |
|----------|----------|-------|
| Random Forest (GridSearchCV) | ~48.1 | Lowest error among sklearn models |
| KNN with scaling (GridSearchCV) | ~45.0 | Competitive after MinMaxScaler |
| Linear Regression (sklearn) | ~43.1 | Strong baseline |
| Linear Regression (custom) | ~41.2 | Gradient descent approximation |
| Decision Tree (GridSearchCV) | ~58.7 | Highest error — prone to overfitting |

Key takeaways:
- **Random Forest** delivered the best balance of accuracy and generalization among the tree-based models.
- **Feature scaling** meaningfully improved KNN performance.
- Building regression algorithms from scratch — gradient descent for linear regression and distance-based prediction for KNN — reinforced my understanding of how these models optimize and make predictions at a fundamental level.